In [1]:
# voorbereiding voor 04
import pandas as pd
import numpy as np
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from mechelen_parking import get_default_paths

PATHS = get_default_paths()
DATA_INT = PATHS.root / "data_intermediate"
DATA_RAW = PATHS.root / "data_raw"

df_st  = pd.read_parquet(DATA_INT / "shortterm_cleaned.parquet")
df_lt  = pd.read_parquet(DATA_INT / "longterm_cleaned.parquet")
df_cal = pd.read_parquet(DATA_INT / "calendar_master.parquet")
df_w   = pd.read_parquet(DATA_INT / "weather_cleaned.parquet")
df_loc = pd.read_parquet(DATA_RAW  / "parking_location.parquet")

SEP = "=" * 60

# ══════════════════════════════════════════════════════════════
# 1. SHORTTERM: rijtellingdiscrepantie
#    Ruw: 697.218 → Cleaned: 284.524  (= -59% !)
#    Oorzaak? Gefilterd op frozen_sensor? Andere jaren verwijderd?
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  1. SHORTTERM — Rijtellinganalyse\n{SEP}")

print(f"Totale rijen         : {len(df_st):,}")
print(f"\nRijen per jaar:")
print(df_st["year"].value_counts().sort_index())
print(f"\nRijen per parking_id:")
print(df_st["parking_id"].value_counts().sort_values(ascending=False))
print(f"\nFlag-verdeling:")
print(df_st[["flag_occ_inconsistent","flag_overcapacity","flag_frozen_sensor"]].sum())
print(f"\nZijn frozen_sensor rijen verwijderd of enkel geflagd?")
print(f"  flag_frozen_sensor > 0 aanwezig: {(df_st['flag_frozen_sensor'] > 0).sum()}")

# ══════════════════════════════════════════════════════════════
# 2. JOIN-KEY ANALYSE — datetime types & timezone
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  2. JOIN-KEY DTYPE ANALYSE\n{SEP}")

print("--- rounded_hour (parking) ---")
print(f"  dtype          : {df_st['rounded_hour'].dtype}")
print(f"  timezone-aware : {df_st['rounded_hour'].dt.tz}")
print(f"  min            : {df_st['rounded_hour'].min()}")
print(f"  max            : {df_st['rounded_hour'].max()}")
print(f"  voorbeeld[0]   : {df_st['rounded_hour'].iloc[0]}  (repr: {repr(df_st['rounded_hour'].iloc[0])})")

print("\n--- timestamp (weather) ---")
print(f"  dtype          : {df_w['timestamp'].dtype}")
print(f"  timezone-aware : {df_w['timestamp'].dt.tz}")
print(f"  min            : {df_w['timestamp'].min()}")
print(f"  max            : {df_w['timestamp'].max()}")
print(f"  voorbeeld[0]   : {df_w['timestamp'].iloc[0]}  (repr: {repr(df_w['timestamp'].iloc[0])})")

print("\n--- date_only (parking) ---")
print(f"  dtype          : {df_st['date_only'].dtype}")
print(f"  voorbeeld[0]   : {df_st['date_only'].iloc[0]}  (type: {type(df_st['date_only'].iloc[0])})")
print(f"  python type    : {type(df_st['date_only'].iloc[0])}")

print("\n--- date (calendar) ---")
print(f"  dtype          : {df_cal['date'].dtype}")
print(f"  voorbeeld[0]   : {df_cal['date'].iloc[0]}  (repr: {repr(df_cal['date'].iloc[0])})")

# ══════════════════════════════════════════════════════════════
# 3. WEERDATA — sortering & bereik check
#    Head(2) toont 2024-datums terwijl data 2020 moet starten!
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  3. WEATHER — Sortering, bereik & volledigheid\n{SEP}")

print(f"Tijdsbereik         : {df_w['timestamp'].min()} → {df_w['timestamp'].max()}")
print(f"Is gesorteerd       : {df_w['timestamp'].is_monotonic_increasing}")
print(f"\nEerste 5 timestamps (ongesorteerd!):")
print(df_w["timestamp"].head(5).to_string())
print(f"\nGesorteerde eerste 5:")
print(df_w["timestamp"].sort_values().head(5).to_string())
print(f"\nRijen per jaar (weather):")
print(df_w["timestamp"].dt.year.value_counts().sort_index())

# Volledigheidscheck
df_w_sorted = df_w.sort_values("timestamp")
expected = pd.date_range(df_w_sorted["timestamp"].min(),
                         df_w_sorted["timestamp"].max(), freq="h")
missing  = expected.difference(df_w_sorted["timestamp"])
print(f"\nVerwachte uren      : {len(expected):,}")
print(f"Werkelijke uren     : {len(df_w_sorted):,}")
print(f"Ontbrekende uren    : {len(missing)}")
if len(missing) > 0:
    print(f"Eerste 5 ontbrekend : {list(missing[:5])}")

# ══════════════════════════════════════════════════════════════
# 4. PARKING_ID KRUISCHECK — matchen IDs tussen bestanden?
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  4. PARKING_ID KRUISCHECK\n{SEP}")

ids_st  = set(df_st["parking_id"].astype(str).unique())
ids_lt  = set(df_lt["parking_id"].astype(str).unique())
ids_loc = set(df_loc["parking_id"].astype(str).unique())

print(f"Unieke IDs in shortterm   ({len(ids_st)}) : {sorted(ids_st)}")
print(f"\nUnieke IDs in longterm    ({len(ids_lt)}) : {sorted(ids_lt)}")
print(f"\nUnieke IDs in loc-tabel   ({len(ids_loc)}): {sorted(ids_loc)}")
print(f"\nIn shortterm maar NIET in loc : {sorted(ids_st - ids_loc)}")
print(f"In longterm  maar NIET in loc : {sorted(ids_lt - ids_loc)}")
print(f"In loc maar NIET in shortterm : {sorted(ids_loc - ids_st)}")

print(f"\nVolledige parking_location tabel:")
print(df_loc.to_string())

# ══════════════════════════════════════════════════════════════
# 5. WEEKDAY ENCODING — consistent tussen parking & kalender?
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  5. WEEKDAY ENCODING\n{SEP}")

sample = df_st[["date_only","weekday_int","weekday_name"]].drop_duplicates().head(10)
print("Parking weekday encoding (sample):")
print(sample.to_string(index=False))

print(f"\nKalender weekday encoding (sample):")
print(df_cal[["date","weekday","is_weekend"]].head(10).to_string(index=False))

print(f"\nZijn 0=Maandag in beide? (check op 2019-01-07 = Maandag)")
st_monday  = df_st[df_st["date_only"] == "2019-01-07"]["weekday_int"].unique()
cal_monday = df_cal[df_cal["date"] == pd.Timestamp("2019-01-07")]["weekday"].values
print(f"  Parking weekday_int voor 2019-01-07: {st_monday}")
print(f"  Kalender weekday    voor 2019-01-07: {cal_monday}")

# ══════════════════════════════════════════════════════════════
# 6. OVERLAP TIJDSBEREIKEN — wat overlapt met weather (2020+)?
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  6. TIJDSBEREIK OVERLAP\n{SEP}")

st_min  = df_st["rounded_hour"].min()
st_max  = df_st["rounded_hour"].max()
lt_min  = df_lt["rounded_hour"].min()
lt_max  = df_lt["rounded_hour"].max()
w_min   = df_w["timestamp"].min()
w_max   = df_w["timestamp"].max()
cal_min = df_cal["date"].min()
cal_max = df_cal["date"].max()

print(f"shortterm   : {st_min}  →  {st_max}")
print(f"longterm    : {lt_min}  →  {lt_max}")
print(f"weather     : {w_min}  →  {w_max}")
print(f"calendar    : {cal_min}  →  {cal_max}")

overlap_start = max(st_min, w_min)
overlap_end   = min(st_max, w_max)
print(f"\nOverlap shortterm ∩ weather : {overlap_start}  →  {overlap_end}")

st_pre_weather = (df_st["rounded_hour"] < w_min).sum()
print(f"\nShortterm rijen VÓÓR weerdata start (krijgen NaN): {st_pre_weather:,} "
      f"({st_pre_weather/len(df_st)*100:.1f}%)")

# ══════════════════════════════════════════════════════════════
# 7. WEATHER AFGELEIDE FEATURES — aanwezig of niet?
#    (is_raining, is_cold, etc. uit notebook 03 stap 7)
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  7. WEATHER — Afgeleide features check\n{SEP}")

derived = ["is_raining","is_heavy_rain","is_freezing","is_cold",
           "is_warm","is_sunny","is_high_wind","temp_c_sq"]
for col in derived:
    present = col in df_w.columns
    print(f"  {col:<20} : {'✓ aanwezig' if present else '✗ ONTBREEKT — aanmaken in nb04'}")

# ══════════════════════════════════════════════════════════════
# 8. CALENDAR — NaN-strings check
#    Zijn NaN-waarden echte NaN of de string "NaN"?
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  8. CALENDAR — NaN vs string-NaN check\n{SEP}")

for col in ["national_holiday_name","other_holiday_name","vacation_name","vacation_type"]:
    n_real_nan   = df_cal[col].isna().sum()
    n_string_nan = (df_cal[col] == "NaN").sum() if df_cal[col].dtype == object else 0
    print(f"  {col:<28}: real NaN={n_real_nan}, string 'NaN'={n_string_nan}")

print(f"\nCalendar day_class verdeling:")
print(df_cal["calendar_day_class"].value_counts())

# ══════════════════════════════════════════════════════════════
# 9. MAD GROOTTE-SCHATTING
# ══════════════════════════════════════════════════════════════
print(f"\n{SEP}\n  9. MAD GROOTTE-SCHATTING\n{SEP}")

n_st      = len(df_st)
n_lt      = len(df_lt)
n_weather = len(df_w)
n_cal     = len(df_cal)

print(f"Shortterm rijen        : {n_st:>8,}")
print(f"Longterm rijen         : {n_lt:>8,}")
print(f"Weather uren           : {n_weather:>8,}")
print(f"Kalender dagen         : {n_cal:>8,}")
print(f"\nMAD shortterm (na joins, 1:1): ~{n_st:,} rijen")
print(f"MAD longterm  (na joins, 1:1): ~{n_lt:,} rijen")
print(f"\nGeschatte extra kolommen na alle joins:")
print(f"  + weather  : {len(df_w.columns) - 1} kolommen  (alle behalve timestamp)")
print(f"  + calendar : {len(df_cal.columns) - 1} kolommen (alle behalve date)")
print(f"  + location : {len(df_loc.columns) - 1} kolommen (alle behalve parking_id)")
print(f"  Huidig     : 21 kolommen (shortterm)")
weather_extra  = len(df_w.columns) - 1
calendar_extra = len(df_cal.columns) - 1
location_extra = len(df_loc.columns) - 1
total = 21 + weather_extra + calendar_extra + location_extra
print(f"  Totaal MAD : ~{total} kolommen")



  1. SHORTTERM — Rijtellinganalyse
Totale rijen         : 284,524

Rijen per jaar:
year
2018        5
2019    33000
2020    32741
2021    13630
2022    12762
2023    39980
2024    57116
2025    87600
2026     7690
Name: count, dtype: int64

Rijen per parking_id:
parking_id
P Bruul                              40196
P Tinel                              37574
P Grote Markt                        36649
P Lamot                              35606
P Veemarkt                           35370
P Keerdok                            26339
P Hoogstraat                         22483
P Kathedraal                         22179
P Maarten                            15473
P Komet                              12655
Bike Halte Mechelen-Veemarkt             0
Bike Station Mechelen                    0
Bike Station Mechelen (Arsenaal)         0
Bike Station Mechelen-Nekkerspoel        0
P Zandpoortvest                          0
Name: count, dtype: int64

Flag-verdeling:
flag_occ_inconsistent        0
flag_o

# 04 — Master Analytical Dataset (MAD) Assembly
## Masterproef: Spatiotemporal Prediction and Optimization of Car Parking in Mechelen

---

### Doel van dit notebook

Dit notebook assembleert de **Master Analytical Dataset (MAD)** door alle
gecleande tussenbestanden samen te voegen tot één coherente analytische dataset.
De MAD is de **enige bron van waarheid** voor alle downstream analyses:
EDA (nb05–07), feature engineering (nb08), modellering (nb09–11),
SHAP-analyse (nb12) en beleidsimulatie (nb13).

**Inputs** (alle uit `data_intermediate/` tenzij anders vermeld):
| Bestand | Rijen | Sleutel |
|---|---|---|
| `shortterm_cleaned.parquet` | 284.524 | `rounded_hour` × `parking_id` |
| `longterm_cleaned.parquet` | 46.643 | `rounded_hour` × `parking_id` |
| `weather_cleaned.parquet` | 52.632 | `timestamp` (uurlijks) |
| `calendar_master.parquet` | 2.922 | `date` (dagelijks) |
| `parking_location.parquet` | 15 | `parking_id` |

**Outputs** (naar `data_processed/`):
| Bestand | Verwachte rijen | Verwachte kolommen |
|---|---|---|
| `MAD_shortterm.parquet` | 284.524 | ~52 |
| `MAD_longterm.parquet` | 46.643 | ~52 |

---

### Architectuur van de joins

shortterm_cleaned ──LEFT JOIN──► × weather (op rounded_hour = timestamp)
──LEFT JOIN──► × calendar (op date_only = date)
──LEFT JOIN──► × location (op parking_id)
══════════════► MAD_shortterm


Alle joins zijn **LEFT JOIN**: elke rij in de parkeerdata blijft behouden,
ook als er geen overeenkomstige weerobservatie of kalenderdag is.
Dit is de enige correcte keuze omdat missing weather (pre-2020) een 
documenteerbare maar aanvaardbare beperking is, geen reden tot uitsluiting
van parkeerobservaties (Saunders et al., 2008).

---

### Bekende datatechnische complicaties (gedocumenteerd in diagnostiek)

1. `rounded_hour` is `datetime64[ns]` (parking) vs `datetime64[us]` (weather)
   → normaliseren naar `datetime64[us]` vóór merge
2. `date_only` is `object` dtype met `datetime.date`-objecten
   → converteren naar `datetime64[us]` vóór calendar-join
3. `weather_cleaned` is **niet gesorteerd** op timestamp
4. `parking_id` is `category` dtype in parking, `str` in location
5. Calendar bevat redundante temporele kolommen (`year`, `month`, `weekday`,
   `is_weekend`) die al aanwezig zijn in de parkeerdata


In [2]:
# imports en paden
import pandas as pd
import numpy as np
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from mechelen_parking import get_default_paths

PATHS = get_default_paths()
ROOT = PATHS.root
DATA_INT = ROOT / "data_intermediate"
DATA_RAW = ROOT / "data_raw"
DATA_PROC = PATHS.data_processed
DATA_PROC.mkdir(parents=True, exist_ok=True)

print(f"DATA_INT  : {DATA_INT}")
print(f"DATA_PROC : {DATA_PROC}")


DATA_INT  : /Users/emilevandevoorde/Documents/mechelen_parking/data_intermediate
DATA_PROC : /Users/emilevandevoorde/Documents/mechelen_parking/data_processed


In [3]:
# alle tussentijdse bestanden inlezen
df_st  = pd.read_parquet(DATA_INT / "shortterm_cleaned.parquet")
df_lt  = pd.read_parquet(DATA_INT / "longterm_cleaned.parquet")
df_w   = pd.read_parquet(DATA_INT / "weather_cleaned.parquet")
df_cal = pd.read_parquet(DATA_INT / "calendar_master.parquet")
df_loc = pd.read_parquet(DATA_RAW  / "parking_location.parquet")

print("Geladen bestanden:")
print(f"  shortterm  : {df_st.shape}")
print(f"  longterm   : {df_lt.shape}")
print(f"  weather    : {df_w.shape}")
print(f"  calendar   : {df_cal.shape}")
print(f"  location   : {df_loc.shape}")


Geladen bestanden:
  shortterm  : (284524, 21)
  longterm   : (46643, 20)
  weather    : (52632, 19)
  calendar   : (2922, 14)
  location   : (15, 6)


## Stap 1 — Pre-merge voorbereiding: dtype-normalisatie

Vóór enige merge wordt uitgevoerd, lossen we de bekende dtype-incompatibiliteiten
op. Dit is een **kritische stap**: een stille dtype-mismatch in Pandas
resulteert niet in een foutmelding maar in een volledig lege join
(alle rijen worden NaN), wat moeilijk te detecteren is zonder expliciete
merge-kwaliteitscontrole.

### Fixes die nodig zijn:

**1. Datetime-precisie** (`ns` → `us`):
Pandas ≥ 2.0 converteert `datetime64[ns]` en `datetime64[us]` automatisch
bij merges, maar expliciete normalisatie voorkomt afhankelijkheid van
versiegedrag en maakt de intent duidelijk.

**2. `date_only`** (`datetime.date` → `datetime64[us]`):
De parkeerdata bevat Python `datetime.date`-objecten in een object-kolom.
Direct joinen op `df_cal["date"]` (datetime64[us]) vereist type-uniformiteit.

**3. `parking_id`** (`category` → `str`):
De locatietabel gebruikt `str`-dtype. Pandas merge met `category` vs `str`
werkt wel, maar expliciete cast elimineert eventuele randgevallen.

**4. Weather sorteren**:
`weather_cleaned` is niet gesorteerd op timestamp (gedocumenteerd in
diagnostiek). Sortering is niet noodzakelijk voor een hash-based merge,
maar is een hygiëne-maatregel en vereist voor eventuele toekomstige
`merge_asof`-operaties.

**5. Calendar: redundante kolommen verwijderen**:
De kalender bevat `year`, `month`, `weekday`, `is_weekend` die al aanwezig
zijn in de parkeerdata. Ze droppen vóór de merge vermijdt `_x`/`_y` suffixen
en houdt de MAD overzichtelijk.


In [4]:
# ── Fix 1 & 2: Datetime-precisie normaliseren ─────────────────────────────────
for df, name in [(df_st, "shortterm"), (df_lt, "longterm")]:
    df["rounded_hour"] = df["rounded_hour"].astype("datetime64[us]")
    df["date_only"]    = pd.to_datetime(df["date_only"]).astype("datetime64[us]")

df_w = df_w.sort_values("timestamp").reset_index(drop=True)

print("Dtype-verificatie na normalisatie:")
print(f"  rounded_hour (st)  : {df_st['rounded_hour'].dtype}")
print(f"  rounded_hour (lt)  : {df_lt['rounded_hour'].dtype}")
print(f"  timestamp (weather): {df_w['timestamp'].dtype}")
print(f"  date_only (st)     : {df_st['date_only'].dtype}")
print(f"  date (calendar)    : {df_cal['date'].dtype}")

# ── Fix 3: parking_id category → str ─────────────────────────────────────────
df_st["parking_id"] = df_st["parking_id"].astype(str)
df_lt["parking_id"] = df_lt["parking_id"].astype(str)

# ── Fix 4: Redundante kalenderkolommen verwijderen ────────────────────────────
# year, month, weekday, is_weekend: al aanwezig in parkeerdata
CAL_DROP = ["year", "month", "weekday", "is_weekend"]
df_cal_clean = df_cal.drop(columns=CAL_DROP)

print(f"\nKalenderkolommen na opruiming ({len(df_cal_clean.columns)}):")
print(df_cal_clean.columns.tolist())


Dtype-verificatie na normalisatie:
  rounded_hour (st)  : datetime64[us]
  rounded_hour (lt)  : datetime64[us]
  timestamp (weather): datetime64[us]
  date_only (st)     : datetime64[us]
  date (calendar)    : datetime64[us]

Kalenderkolommen na opruiming (10):
['date', 'national_holiday_name', 'other_holiday_name', 'vacation_name', 'vacation_type', 'is_national_holiday', 'is_other_holiday', 'is_any_holiday', 'is_school_vacation', 'calendar_day_class']


## Stap 2 — Locatietabel voorbereiden

De locatietabel bevat 15 parkings, waarvan:
- **10 autoparkings**: hebben tijdreeksdata in shortterm (P Zandpoortvest ontbreekt!)
- **1 autoparking zonder data**: P Zandpoortvest (622 plaatsen, vesten-categorie)
  → staat wel in locatietabel maar heeft **geen rijen** in de tijdreeksen
- **4 fietsstations**: geen tijdreeksdata

P Zandpoortvest is een belangrijke beperking voor de analyse: de grootste
parking in de Vesten-categorie (622/1.409 plaatsen = 44% van vestencapaciteit)
is niet vertegenwoordigd in het model. Dit dient expliciet gedocumenteerd
te worden in de thesis-methodologie als **data-beschikbaarheidsbeperking**.

We droppen `created_at` uit de locatietabel (administratieve metadata).


In [5]:
# locatietabel voorbereiden
df_loc_clean = df_loc.drop(columns=["created_at"]).copy()

# Lowercase categorie voor consistentie (loc heeft 'centrum'/'vesten'/'rand')
df_loc_clean["parking_location_category"] = (
    df_loc_clean["parking_location_category"].astype(str)
)

print("Locatietabel na opruiming:")
print(df_loc_clean.to_string(index=False))

# ── Documenteer P Zandpoortvest beperking ─────────────────────────────────────
missing_in_st = set(df_loc_clean["parking_id"]) - set(df_st["parking_id"])
print(f"\nParkings in locatietabel maar NIET in shortterm:")
for pid in sorted(missing_in_st):
    row = df_loc_clean[df_loc_clean["parking_id"] == pid]
    cat = row["parking_location_category"].values[0]
    cap = row["total_capacity"].values[0]
    print(f"  ⚠️  {pid:<40} cat={cat}  cap={cap}")


Locatietabel na opruiming:
                       parking_id  longitude  latitude  total_capacity parking_location_category
     Bike Halte Mechelen-Veemarkt   4.484340 51.029300             NaN                       NaN
            Bike Station Mechelen   4.483001 51.017822             NaN                       NaN
 Bike Station Mechelen (Arsenaal)   4.483918 51.016306             NaN                       NaN
Bike Station Mechelen-Nekkerspoel   4.489779 51.030347             NaN                       NaN
                          P Bruul   4.484515 51.021180           350.0                    vesten
                    P Grote Markt   4.479361 51.028225           155.0                   centrum
                     P Hoogstraat   4.474746 51.023465           109.0                   centrum
                     P Kathedraal   4.477972 51.029510           130.0                   centrum
                        P Keerdok   4.470091 51.034707           516.0                      rand
   

### Stap 2b — Locatietabel: twee resterende problemen

#### Probleem 1: `astype(str)` converteert NaN → string `"nan"`

In cel 7 werd `parking_location_category.astype(str)` uitgevoerd om het
`category`-dtype te normaliseren. Dit heeft als neveneffect dat echte NaN-waarden
(voor fietsstations) omgezet worden naar de letterlijke string `"nan"`.
Bij een eventuele `groupby` of `value_counts` zou `"nan"` als aparte categorie
verschijnen — een stille, moeilijk te detecteren fout.

Oplossing: **bewaar het originele dtype** (`category`) en converteer enkel
naar `str` waar effectief nodig (bv. in de merge-stap zelf via `.astype(str)`
op de join-sleutel).

#### Probleem 2: Fietsstations in de locatietabel

De 4 fietsstations hebben `NaN` voor `total_capacity` en
`parking_location_category`. Omdat de MAD uitsluitend over autoparkings gaat
en geen van deze IDs voorkomt in de parking-tijdreeksen, zijn ze **functioneel
irrelevant** voor de joins. Toch is het hygiënischer om de locatietabel te
beperken tot entiteiten die effectief in de MAD kunnen voorkomen:

- Autoverkeer-gerelateerde parkings: de 10 unieke `parking_id`s uit de tijdreeksen
  + P Zandpoortvest (aanwezig in locatietabel, maar geen tijdreeksdata → bewaren
  voor geografische volledigheid in cartografische analyses)

Fietsstations worden **uitgesloten** uit `df_loc_clean` omdat ze geen
informatiewaarde toevoegen aan de MAD en NaN-vervuiling veroorzaken.


In [6]:
# ── Fix 1: parking_location_category terug naar category dtype ────────────────
# (astype(str) in cel 7 was te agressief — maak opnieuw correct)
df_loc_clean["parking_location_category"] = (
    df_loc["parking_location_category"]   # originele categorie-kolom
    # blijft category dtype; NaN blijft NaN
)

# ── Fix 2: Fietsstations uitsluiten ────────────────────────────────────────────
# Definieer inclusiecriterium: alleen rijen met een geldige categorie
# (fietsstations hebben NaN voor parking_location_category)
mask_car_parking = df_loc_clean["parking_location_category"].notna()
df_loc_clean     = df_loc_clean[mask_car_parking].copy()

print(f"Locatietabel na filtering: {len(df_loc_clean)} rijen  "
      f"(was {len(df_loc)}, {len(df_loc) - len(df_loc_clean)} fietsstations verwijderd)")

print(f"\nVerificatie dtype parking_location_category: "
      f"{df_loc_clean['parking_location_category'].dtype}")

print(f"\nFinale locatietabel die gebruikt wordt in alle joins:")
print(df_loc_clean.to_string(index=False))

# ── Controle: zijn alle parking-IDs uit de tijdreeksen aanwezig? ──────────────
ids_tijdreeksen = set(df_st["parking_id"]) | set(df_lt["parking_id"])
ids_loc_clean   = set(df_loc_clean["parking_id"])

in_data_not_loc = ids_tijdreeksen - ids_loc_clean
in_loc_not_data = ids_loc_clean   - ids_tijdreeksen

print(f"\nIn tijdreeksen maar NIET in df_loc_clean : {sorted(in_data_not_loc)}")
print(f"In df_loc_clean maar NIET in tijdreeksen  : {sorted(in_loc_not_data)}")

if not in_data_not_loc:
    print("\n✓ Alle parking-IDs uit de tijdreeksen zijn aanwezig in df_loc_clean.")
    print("  (P Zandpoortvest aanwezig in df_loc_clean voor geografische analyses.)")
else:
    print(f"\n⚠️  {len(in_data_not_loc)} parking-IDs zonder locatiedata!")


Locatietabel na filtering: 11 rijen  (was 15, 4 fietsstations verwijderd)

Verificatie dtype parking_location_category: category

Finale locatietabel die gebruikt wordt in alle joins:
     parking_id  longitude  latitude  total_capacity parking_location_category
        P Bruul   4.484515 51.021180           350.0                    vesten
  P Grote Markt   4.479361 51.028225           155.0                   centrum
   P Hoogstraat   4.474746 51.023465           109.0                   centrum
   P Kathedraal   4.477972 51.029510           130.0                   centrum
      P Keerdok   4.470091 51.034707           516.0                      rand
        P Komet   4.470464 51.024324           124.0                    vesten
        P Lamot   4.476811 51.025404           255.0                   centrum
      P Maarten   4.477091 51.020229           189.0                    vesten
        P Tinel   4.482241 51.033469           124.0                    vesten
     P Veemarkt   4.483625

## Stap 3 — Join-strategie

### Merge-volgorde en sleutels

| Stap | Join | Linkersleutel | Rechtersleutel | Type |
|---|---|---|---|---|
| A | parking × weather | `rounded_hour` | `timestamp` | LEFT |
| B | result A × calendar | `date_only` | `date` | LEFT |
| C | result B × location | `parking_id` | `parking_id` | LEFT |

### Verwachte NaN-patronen na joins

**Join A (weather)**:
- 33.005 rijen (11.6%) vóór 2020-01-01 → NaN voor alle 18 weerkolommen
- Rijen na 2026-01-01 23:00 (max weather) → NaN (7.690 rijen, jaar 2026)
- Rijen met `flag_frozen_sensor=1` behouden hun weerdata (sensor-uitval
  in parking ≠ sensor-uitval in weerstation)

**Join B (calendar)**:
- 5 rijen met `date_only = 2018-12-31` vallen buiten calendar (start 2019-01-01)
  → NaN voor kalenderkolommen

**Join C (location)**:
- Alle 10 parking_ids in shortterm zitten in de locatietabel → 0 NaN verwacht

### Hantering van NaN-tekstkolommen in de kalender
`national_holiday_name`, `other_holiday_name`, `vacation_name`, `vacation_type`
zijn NaN voor gewone dagen — dit is **correct** (geen fout), niet te imputeren.


In [7]:
# MAD shortterm — Join A (weather)
n_before = len(df_st)

MAD_st = df_st.merge(
    df_w.rename(columns={"timestamp": "rounded_hour"}),
    on="rounded_hour",
    how="left",
    validate="many_to_one"   # meerdere parkings per uur, 1 weerobs per uur
)

n_after = len(MAD_st)
assert n_before == n_after, f"Rijen verloren bij weather join: {n_before} → {n_after}"

# ── Merge-kwaliteit ───────────────────────────────────────────────────────────
n_nan_weather = MAD_st["temp_c"].isna().sum()
n_pre2020     = (MAD_st["rounded_hour"] < pd.Timestamp("2020-01-01")).sum()
n_post_w_max  = (MAD_st["rounded_hour"] > pd.Timestamp("2026-01-01 23:00")).sum()

print(f"Join A (weather) — shortterm:")
print(f"  Rijen vóór  : {n_before:,}")
print(f"  Rijen na    : {n_after:,}  ✓")
print(f"  NaN temp_c  : {n_nan_weather:,}")
print(f"    → pre-2020                  : {n_pre2020:,} rijen (verwacht ~33.005)")
print(f"    → post weather-max          : {n_post_w_max:,} rijen")
print(f"    → onverklaard               : {n_nan_weather - n_pre2020 - n_post_w_max:,} rijen")


Join A (weather) — shortterm:
  Rijen vóór  : 284,524
  Rijen na    : 284,524  ✓
  NaN temp_c  : 40,455
    → pre-2020                  : 33,005 rijen (verwacht ~33.005)
    → post weather-max          : 7,450 rijen
    → onverklaard               : 0 rijen


In [8]:
# MAD shortterm — Join B (calendar)
MAD_st = MAD_st.merge(
    df_cal_clean.rename(columns={"date": "date_only"}),
    on="date_only",
    how="left",
    validate="many_to_one"
)

assert len(MAD_st) == n_before, f"Rijen verloren bij calendar join"

n_nan_cal = MAD_st["calendar_day_class"].isna().sum()
print(f"Join B (calendar) — shortterm:")
print(f"  Rijen na    : {len(MAD_st):,}  ✓")
print(f"  NaN calendar_day_class: {n_nan_cal:,}  (verwacht: 5 rijen uit 2018-12-31)")

# Calendar day_class verdeling in MAD
print(f"\\nCalendar day_class verdeling in MAD:")
print(MAD_st["calendar_day_class"].value_counts(dropna=False))


Join B (calendar) — shortterm:
  Rijen na    : 284,524  ✓
  NaN calendar_day_class: 5  (verwacht: 5 rijen uit 2018-12-31)
\nCalendar day_class verdeling in MAD:
calendar_day_class
regular_day         207577
school_vacation      63708
national_holiday      8076
other_holiday         5158
NaN                      5
Name: count, dtype: int64


In [9]:
# MAD shortterm — Join C (location)
MAD_st = MAD_st.merge(
    df_loc_clean,
    on="parking_id",
    how="left",
    validate="many_to_one"
)

assert len(MAD_st) == n_before, f"Rijen verloren bij location join"

n_nan_loc = MAD_st["parking_location_category"].isna().sum()
print(f"Join C (location) — shortterm:")
print(f"  Rijen na    : {len(MAD_st):,}  ✓")
print(f"  NaN parking_location_category: {n_nan_loc:,}  (verwacht: 0)")
print(f"\\nLocatiecategorie-verdeling:")
print(MAD_st["parking_location_category"].value_counts())


Join C (location) — shortterm:
  Rijen na    : 284,524  ✓
  NaN parking_location_category: 0  (verwacht: 0)
\nLocatiecategorie-verdeling:
parking_location_category
centrum    152287
vesten     105898
rand        26339
Name: count, dtype: int64


In [10]:
# MAD longterm — alle joins
n_lt_before = len(df_lt)

# ── Join A: weather ────────────────────────────────────────────────────────────
MAD_lt = df_lt.merge(
    df_w.rename(columns={"timestamp": "rounded_hour"}),
    on="rounded_hour",
    how="left",
    validate="many_to_one"
)

# ── Join B: calendar ───────────────────────────────────────────────────────────
MAD_lt = MAD_lt.merge(
    df_cal_clean.rename(columns={"date": "date_only"}),
    on="date_only",
    how="left",
    validate="many_to_one"
)

# ── Join C: location ───────────────────────────────────────────────────────────
MAD_lt = MAD_lt.merge(
    df_loc_clean,
    on="parking_id",
    how="left",
    validate="many_to_one"
)

assert len(MAD_lt) == n_lt_before, "Rijen verloren bij longterm joins"

print(f"MAD longterm: {MAD_lt.shape}")
print(f"  NaN temp_c                   : {MAD_lt['temp_c'].isna().sum():,}  (verwacht: 0, alles 2024)")
print(f"  NaN calendar_day_class       : {MAD_lt['calendar_day_class'].isna().sum():,}")
print(f"  NaN parking_location_category: {MAD_lt['parking_location_category'].isna().sum():,}")


MAD longterm: (46643, 51)
  NaN temp_c                   : 0  (verwacht: 0, alles 2024)
  NaN calendar_day_class       : 0
  NaN parking_location_category: 0


## Stap 4 — Volledig merge-kwaliteitsrapport

We genereren een systematisch overzicht van alle NaN-patronen in de MAD.
Hiermee documenteren we expliciet welke kolommen NaN bevatten, hoeveel,
en of dat **verwacht** (structureel) of **onverwacht** (probleem) is.

Verwachte NaN-structuur voor MAD shortterm:

| Kolomgroep | Verwachte NaN | Oorzaak |
|---|---|---|
| Weerkolommen | ~33.005 + ~7.690 | Pre-2020 en post-2026-01 rijen |
| Kalenderkolommen (namen) | ~5 | 2018-12-31 buiten calenderbereik |
| `calendar_day_class` | ~5 | Idem |
| Locatiekolommen | 0 | Alle 10 parking_ids gematch |


In [11]:
def merge_quality_report(df: pd.DataFrame, name: str):
    print(f"\n{'='*60}")
    print(f"  MERGE-KWALITEITSRAPPORT — {name}")
    print(f"{'='*60}")
    print(f"  Shape     : {df.shape}")
    print(f"  Rijen     : {len(df):,}")
    print(f"  Kolommen  : {len(df.columns)}")

    nan_counts = df.isnull().sum()
    nan_pct    = nan_counts / len(df) * 100
    nan_df     = pd.DataFrame({
        "n_nan" : nan_counts,
        "pct"   : nan_pct.round(2),
        "status": ""
    })

    # Classificeer als verwacht / onverwacht
    EXPECTED_NAN = [
        "temp_c","precip_mm","wind_speed_ms","wind_gusts_ms","humidity_pct",
        "pressure_hpa","sun_duration_min","shortwave_wm2","sun_intensity_wm2",
        "qc_temp","qc_precip","qc_wind_speed","qc_wind_gusts","qc_humidity",
        "qc_pressure","humidity_suspect","sun_duration_inconsistent","precip_imputed",
        "national_holiday_name","other_holiday_name","vacation_name","vacation_type",
        "calendar_day_class"
    ]
    for col in nan_df.index:
        if nan_df.loc[col, "n_nan"] == 0:
            nan_df.loc[col, "status"] = "✓ compleet"
        elif col in EXPECTED_NAN:
            nan_df.loc[col, "status"] = "○ verwacht"
        else:
            nan_df.loc[col, "status"] = "⚠️  ONVERWACHT"

    print(f"\n  Kolommen met NaN:")
    has_nan = nan_df[nan_df["n_nan"] > 0]
    for col, row in has_nan.iterrows():
        print(f"    {col:<35} {row['n_nan']:>7,}  ({row['pct']:>5.1f}%)  {row['status']}")

    unexpected = nan_df[(nan_df["n_nan"] > 0) & (nan_df["status"].str.contains("ONVERWACHT"))]
    if len(unexpected) > 0:
        print(f"\n  ⚠️  {len(unexpected)} ONVERWACHTE NaN-kolommen — onderzoek vereist!")
    else:
        print(f"\n  ✓ Alle NaN-waarden zijn structureel verwacht.")

merge_quality_report(MAD_st, "MAD_shortterm")
merge_quality_report(MAD_lt, "MAD_longterm")



  MERGE-KWALITEITSRAPPORT — MAD_shortterm
  Shape     : (284524, 52)
  Rijen     : 284,524
  Kolommen  : 52

  Kolommen met NaN:
    temp_c                               40,455  ( 14.2%)  ○ verwacht
    precip_mm                            40,455  ( 14.2%)  ○ verwacht
    wind_speed_ms                        40,455  ( 14.2%)  ○ verwacht
    wind_gusts_ms                        40,455  ( 14.2%)  ○ verwacht
    humidity_pct                         40,455  ( 14.2%)  ○ verwacht
    pressure_hpa                         40,455  ( 14.2%)  ○ verwacht
    sun_duration_min                     40,455  ( 14.2%)  ○ verwacht
    shortwave_wm2                        40,455  ( 14.2%)  ○ verwacht
    sun_intensity_wm2                    40,455  ( 14.2%)  ○ verwacht
    qc_temp                              40,455  ( 14.2%)  ○ verwacht
    qc_precip                            40,455  ( 14.2%)  ○ verwacht
    qc_wind_speed                        40,455  ( 14.2%)  ○ verwacht
    qc_wind_gusts             

## Stap 5 — Administratieve kolommen beoordelen

De MAD bevat na de joins ~57 kolommen. Sommige zijn puur administratief
en voegen geen analytische waarde toe. We documenteren de beslissing
per kolom — niets wordt stilzwijgend verwijderd.

| Kolom | Beslissing | Motivering |
|---|---|---|
| `parking_id_hash` | **Bewaren** | Kan gebruikt worden voor geanonimiseerde rapportage |
| `publication_time` | **Droppen** | Administratieve timestamp; `rounded_hour` is de canonieke tijdsleutel |
| `created_at` | **Droppen** | Systeemtimestamp van data-ingestie; geen analytische waarde |
| `date_with_day` | **Droppen** | Redundant: combinatie van `date_only` + `weekday_name` |
| `parking_type` | **Bewaren** | Altijd "Car Parking" in shortterm, maar bewaren voor volledigheid |
| `kind` | **Bewaren** | Onderscheidt shortterm/longterm — relevant bij eventuele samenvoeging |

Na deze opruiming bereiken we een **lean MAD** zonder informatieverlies.


In [12]:
# Administratieve kolommen verwijderen
DROP_ADMIN = ["publication_time", "created_at", "date_with_day"]

MAD_st = MAD_st.drop(columns=DROP_ADMIN, errors="ignore")
MAD_lt = MAD_lt.drop(columns=DROP_ADMIN, errors="ignore")

print(f"MAD shortterm na opruiming : {MAD_st.shape}")
print(f"MAD longterm  na opruiming : {MAD_lt.shape}")

print(f"\nFinale kolommen MAD shortterm ({len(MAD_st.columns)}):")
for i, col in enumerate(MAD_st.columns, 1):
    print(f"  {i:>2}. {col}")



MAD shortterm na opruiming : (284524, 49)
MAD longterm  na opruiming : (46643, 48)

Finale kolommen MAD shortterm (49):
   1. parking_id
   2. parking_id_hash
   3. parking_type
   4. kind
   5. year
   6. month
   7. date_only
   8. rounded_hour
   9. hour
  10. weekday_int
  11. weekday_name
  12. day_type
  13. number_of_spaces_override
  14. number_of_occupied_spaces
  15. occupancy_rate
  16. flag_occ_inconsistent
  17. flag_overcapacity
  18. flag_frozen_sensor
  19. temp_c
  20. precip_mm
  21. wind_speed_ms
  22. wind_gusts_ms
  23. humidity_pct
  24. pressure_hpa
  25. sun_duration_min
  26. shortwave_wm2
  27. sun_intensity_wm2
  28. qc_temp
  29. qc_precip
  30. qc_wind_speed
  31. qc_wind_gusts
  32. qc_humidity
  33. qc_pressure
  34. humidity_suspect
  35. sun_duration_inconsistent
  36. precip_imputed
  37. national_holiday_name
  38. other_holiday_name
  39. vacation_name
  40. vacation_type
  41. is_national_holiday
  42. is_other_holiday
  43. is_any_holiday
  44. is_

## Stap 6 — Datadekking & structurele databeperkingen

Voordat de MAD geëxporteerd wordt, voeren we een systematische
dekkingsanalyse uit om structurele databeperkingen te identificeren,
te kwantificeren en te flaggen. Databeperkingen worden **niet gefixed**
maar transparant gedocumenteerd en als binaire flags bewaard — conform
het principe van *non-destructive preprocessing*: ruwe informatie
wordt nooit permanent verwijderd, zodat elke downstream analyse zelf
kan beslissen welke rijen te gebruiken (Saunders et al., 2008).

Drie flags worden aangemaakt:

| Flag | Type | Definitie |
|---|---|---|
| `low_data_coverage` | int8 (0/1) | Jaren met < 50% jaarlijkse dekking op jaarniveau (2018, 2021, 2022) |
| `system_blackout` | int8 (0/1) | Rijen die vallen in de empirisch vastgestelde uitvalperiode (jun 2021 – jul 2022) |
| `partial_year` | int8 (0/1) | Lopend jaar (2026): temporeel onvolledig, géén dekkingsprobleem |

Het onderscheid tussen `low_data_coverage` en `system_blackout` is
bewust: `low_data_coverage` werkt op jaarniveau en is snel filterbaar;
`system_blackout` is preciezer (maandniveau) en capteert de sparse
observaties die wél bestonden tijdens de uitvalperiode.


In [23]:
# ── 1. Dekking per jaar (jaar-niveau) ─────────────────────────────────────────
days_per_year = {2018:365, 2019:365, 2020:366, 2021:365,
                 2022:365, 2023:365, 2024:366, 2025:365, 2026:365}

print("=== Dekking per jaar (shortterm) ===")
year_counts = MAD_st.groupby("year")["rounded_hour"].count()
for year, count in year_counts.items():
    n_pid    = MAD_st[MAD_st["year"] == year]["parking_id"].nunique()
    expected = n_pid * days_per_year[year] * 24
    coverage = count / expected * 100 if expected > 0 else 0
    flag     = "  ⚠️" if coverage < 50 else ""
    print(f"  {year}: {count:>7,} / {expected:>7,} verwacht  "
          f"({coverage:>5.1f}%  op {n_pid:>2} parkings){flag}")

# ── 2. Dekking per parking per jaar (heatmap in tabelvorm) ─────────────────────
print("\n=== Dekking (%) per parking per jaar ===")
pivot_counts = (MAD_st
                .groupby(["year", "parking_id"])["rounded_hour"]
                .count()
                .unstack(fill_value=0))

pivot_pct = pivot_counts.astype(float).copy()
for year in pivot_pct.index:
    pivot_pct.loc[year] = (
        pivot_counts.loc[year] / (days_per_year[year] * 24) * 100
    ).round(1)
print(pivot_pct.to_string())

# ── 3. Zoom: 2021–2022 per maand per parking ──────────────────────────────────
print("\n=== Detail 2021–2022: rijen per maand per parking ===")
for year in [2021, 2022]:
    print(f"\n  Jaar {year}:")
    monthly = (MAD_st[MAD_st["year"] == year]
               .groupby(["month", "parking_id"])["rounded_hour"]
               .count()
               .unstack(fill_value=0))
    print(monthly.to_string())


=== Dekking per jaar (shortterm) ===
  2018:       5 /  43,800 verwacht  (  0.0%  op  5 parkings)  ⚠️
  2019:  33,000 /  52,560 verwacht  ( 62.8%  op  6 parkings)
  2020:  32,741 /  52,704 verwacht  ( 62.1%  op  6 parkings)
  2021:  13,630 /  43,800 verwacht  ( 31.1%  op  5 parkings)  ⚠️
  2022:  12,762 /  52,560 verwacht  ( 24.3%  op  6 parkings)  ⚠️
  2023:  39,980 /  70,080 verwacht  ( 57.0%  op  8 parkings)
  2024:  57,116 /  87,840 verwacht  ( 65.0%  op 10 parkings)
  2025:  87,600 /  87,600 verwacht  (100.0%  op 10 parkings)
  2026:   7,690 /  87,600 verwacht  (  8.8%  op 10 parkings)  ⚠️

=== Dekking (%) per parking per jaar ===
parking_id  P Bruul  P Grote Markt  P Hoogstraat  P Kathedraal  P Keerdok  P Komet  P Lamot  P Maarten  P Tinel  P Veemarkt
year                                                                                                                       
2018            0.0            0.0           0.0           0.0        0.0      0.0      0.0        0.0      

### Interpretatie: twee structureel verschillende databeperkingen

De dekkingsanalyse onthult twee oorzaken die fundamenteel van aard
verschillen. Ze vereisen een gescheiden behandeling, zowel in de
flagging als in de thesis-methodologie.

---

#### Beperking 1 — Systeem-uitval: juni 2021 t/m juni 2022

De data stopt abrupt na mei 2021 (vrijwel alle parkings) en hervat
pas in juli 2022. P Bruul vormt een gedeeltelijke uitzondering met
sporadische observaties tot oktober 2021, maar ook hier is de
continuïteit verbroken.

Dit patroon is kenmerkend voor een **MNAR-situatie** (*Missing Not At
Random*): de missingness is niet willekeurig maar het gevolg van een
systeemgebeurtenis (API-uitval, platformmigratie of archiveringsfout).
MNAR is de meest problematische categorie van missing data voor
statistische modellen, omdat de kans op ontbrekende observaties
gecorreleerd is met het meetproces zelf — niet met de te verklaren
variabele (Little & Rubin, 2002; van Buuren, 2018). Imputatie over
deze periode is methodologisch onverantwoord; uitsluiting is de
enige correcte aanpak.

**Empirisch vastgestelde blackout-periode**: `2021-06-01` t/m `2022-07-01`

---

#### Beperking 2 — Gefaseerde parking-onboarding

Parkings zijn niet simultaan aangesloten op het datasysteem maar
werden successief toegevoegd tussen 2019 en 2024:

| Cohort | Parkings | Eerste stabiele maand (≥ 80% dekking) |
|---|---|---|
| Vroeg (2019) | P Bruul, P Lamot | 2019-01 |
| Midden (2020) | P Grote Markt, P Veemarkt | 2020-01 / 2020-08 |
| Laat (2023) | P Keerdok, P Hoogstraat | 2023-02 / 2023-10 |
| Recent (2024) | P Tinel, P Maarten, P Kathedraal, P Komet | 2024-01 t/m 2024-09 |

Dit resulteert in een **ongebalanceerd paneeldataset** (*unbalanced
panel*): niet elke parking-tijdreeks heeft dezelfde lengte. Voor
tijdreeksmodellen die over alle parkings tegelijk trainen (globale
modellen), is dit een gekende complicatie: het model kan features
leren die enkel beschikbaar zijn voor parkings met lange reeksen
en die de parkings met korte reeksen impliciet bevoordeelt
(Bergmeir & Benítez, 2012; Montero-Manso & Hyndman, 2021).

**Aanbeveling voor modelleerfase**: evalueer modellen zowel op de
volledige trainingsset als op de subset van parkings met ≥ 2 jaar
stabiele data (P Bruul, P Lamot, P Grote Markt, P Veemarkt),
als sensitiviteitscheck op de onbalans.

---

#### Beperking 3 — P Zandpoortvest: grootste vesten-parking afwezig

P Zandpoortvest (622 plaatsen) heeft in geen enkel brondatabestand
tijdreeksdata. Dit vertegenwoordigt 44% van de totale vestencapaciteit
(622 / (350 + 124 + 189 + 124 + 622) = 622/1.409). Hiërarchische
analyses op categorieniveau "vesten" zijn daardoor onvolledig.


In [24]:
# ── Flag 1: low_data_coverage (jaar-niveau: 2018, 2021, 2022) ─────────────────
LOW_COVERAGE_YEARS = {2018, 2021, 2022}
for df in [MAD_st, MAD_lt]:
    df["low_data_coverage"] = df["year"].isin(LOW_COVERAGE_YEARS).astype("int8")

# ── Flag 2: system_blackout (maand-niveau, empirisch vastgesteld) ─────────────
BLACKOUT_START = pd.Timestamp("2021-06-01")
BLACKOUT_END   = pd.Timestamp("2022-07-01")
for df in [MAD_st, MAD_lt]:
    df["system_blackout"] = (
        (df["rounded_hour"] >= BLACKOUT_START) &
        (df["rounded_hour"] <  BLACKOUT_END)
    ).astype("int8")

# ── Flag 3: partial_year (lopend jaar 2026) ────────────────────────────────────
for df in [MAD_st, MAD_lt]:
    df["partial_year"] = (df["year"] == 2026).astype("int8")

# ── Onboarding referentietabel (80%-drempel: ≥ 576 aanwezige uren/maand) ──────
onboarding = {}
for pid in MAD_st["parking_id"].unique():
    monthly = (MAD_st[MAD_st["parking_id"] == pid]
               .groupby(pd.Grouper(key="rounded_hour", freq="ME"))["rounded_hour"]
               .count())
    stable = monthly[monthly >= 576]
    onboarding[pid] = (stable.index.min().strftime("%Y-%m")
                       if len(stable) > 0 else None)

onboard_df = (pd.DataFrame.from_dict(onboarding, orient="index",
                                      columns=["first_stable_month"])
              .sort_values("first_stable_month"))

# Voeg toe aan locatietabel (enkel als nog niet aanwezig)
if "first_stable_month" not in df_loc_clean.columns:
    df_loc_clean = df_loc_clean.merge(
        onboard_df.reset_index().rename(columns={"index": "parking_id"}),
        on="parking_id", how="left"
    )

# ── Overzicht ─────────────────────────────────────────────────────────────────
print("=== Flags — shortterm ===")
for flag in ["low_data_coverage", "system_blackout", "partial_year"]:
    n   = int(MAD_st[flag].sum())
    pct = n / len(MAD_st) * 100
    print(f"  {flag:<22}: {n:>7,}  ({pct:.1f}%)")

print("\n=== Onboarding per parking ===")
print(onboard_df.to_string())


=== Flags — shortterm ===
  low_data_coverage     :  26,397  (9.3%)
  system_blackout       :   2,564  (0.9%)
  partial_year          :   7,690  (2.7%)

=== Onboarding per parking ===
              first_stable_month
P Bruul                  2019-01
P Lamot                  2019-01
P Grote Markt            2020-01
P Veemarkt               2020-08
P Keerdok                2023-02
P Hoogstraat             2023-10
P Tinel                  2024-01
P Maarten                2024-05
P Kathedraal             2024-08
P Komet                  2024-09


In [25]:
# ── Flag-overlap: low_data_coverage ∩ system_blackout ─────────────────────────
overlap      = ((MAD_st["low_data_coverage"]==1) & (MAD_st["system_blackout"]==1)).sum()
only_blackout= ((MAD_st["low_data_coverage"]==0) & (MAD_st["system_blackout"]==1)).sum()
only_lowcov  = ((MAD_st["low_data_coverage"]==1) & (MAD_st["system_blackout"]==0)).sum()

print("=== Flag-overlap (low_data_coverage ∩ system_blackout) ===")
print(f"  Beide flags = 1                      : {overlap:>6,}")
print(f"  Enkel system_blackout = 1            : {only_blackout:>6,}  "
      f"← sparse obs. in blackout buiten 2021/2022")
print(f"  Enkel low_data_coverage = 1          : {only_lowcov:>6,}  "
      f"← lage dekking buiten blackout-periode")

# ── Trainingsset definitie ─────────────────────────────────────────────────────
# Gecombineerd uitsluitingscriterium voor modeltraining
exclude_mask = (
    (MAD_st["low_data_coverage"] == 1) |
    (MAD_st["system_blackout"]   == 1) |
    (MAD_st["partial_year"]      == 1) |
    (MAD_st["year"]              <  2020)   # geen weerdata
)

print("\n=== Trainingsset definitie ===")
print(f"  Totaal MAD shortterm             : {len(MAD_st):>7,}")
print(f"  Uit te sluiten (unie alle flags) : {exclude_mask.sum():>7,}  "
      f"({exclude_mask.mean()*100:.1f}%)")
print(f"  Beschikbaar voor training        : {(~exclude_mask).sum():>7,}  "
      f"({(~exclude_mask).mean()*100:.1f}%)")

train_years = sorted(MAD_st[~exclude_mask]["year"].unique())
print(f"  Jaren in trainingsset            : {train_years}")

# ── Per-jaar breakdown trainingsset ───────────────────────────────────────────
print("\n=== Trainingsset per jaar ===")
print(f"  {'Jaar':<6} {'Rijen':>8}  {'Parkings':>9}  {'Dekking':>9}")
print("  " + "-" * 38)
for year in train_years:
    subset   = MAD_st[(~exclude_mask) & (MAD_st["year"] == year)]
    n_pid    = subset["parking_id"].nunique()
    expected = n_pid * days_per_year.get(year, 365) * 24
    coverage = len(subset) / expected * 100 if expected > 0 else 0
    print(f"  {year:<6} {len(subset):>8,}  {n_pid:>9}  {coverage:>8.1f}%")

n_holdout = MAD_st[MAD_st["year"] == 2025]["rounded_hour"].count()
print(f"\n  2025 holdout (apart)             : {n_holdout:>7,}  (100% dekking, 10 parkings)")


=== Flag-overlap (low_data_coverage ∩ system_blackout) ===
  Beide flags = 1                      :  2,564
  Enkel system_blackout = 1            :      0  ← sparse obs. in blackout buiten 2021/2022
  Enkel low_data_coverage = 1          : 23,833  ← lage dekking buiten blackout-periode

=== Trainingsset definitie ===
  Totaal MAD shortterm             : 284,524
  Uit te sluiten (unie alle flags) :  67,087  (23.6%)
  Beschikbaar voor training        : 217,437  (76.4%)
  Jaren in trainingsset            : [np.int32(2020), np.int32(2023), np.int32(2024), np.int32(2025)]

=== Trainingsset per jaar ===
  Jaar      Rijen   Parkings    Dekking
  --------------------------------------
  2020     32,741          6      62.1%
  2023     39,980          8      57.0%
  2024     57,116         10      65.0%
  2025     87,600         10     100.0%

  2025 holdout (apart)             :  87,600  (100% dekking, 10 parkings)


In [26]:
# ── Export locatietabel met onboarding-info ───────────────────────────────────
loc_out = DATA_INT / "parking_location_clean.parquet"
df_loc_clean.to_parquet(loc_out, index=False)
print(f"✓ parking_location_clean.parquet")
print(f"  Pad    : {loc_out}")
print(f"  Shape  : {df_loc_clean.shape}")
print(f"  Kolommen: {df_loc_clean.columns.tolist()}")

# ── Export MAD shortterm ───────────────────────────────────────────────────────
out_st = DATA_PROC / "MAD_shortterm.parquet"
MAD_st.to_parquet(out_st, index=False)

# ── Export MAD longterm ────────────────────────────────────────────────────────
out_lt = DATA_PROC / "MAD_longterm.parquet"
MAD_lt.to_parquet(out_lt, index=False)

print(f"\n✓ MAD_shortterm.parquet")
print(f"  Pad      : {out_st}")
print(f"  Shape    : {MAD_st.shape}")
print(f"  Grootte  : {out_st.stat().st_size / 1024 / 1024:.1f} MB")
print(f"  Flags    : low_data_coverage, system_blackout, partial_year")

print(f"\n✓ MAD_longterm.parquet")
print(f"  Pad      : {out_lt}")
print(f"  Shape    : {MAD_lt.shape}")
print(f"  Grootte  : {out_lt.stat().st_size / 1024 / 1024:.1f} MB")


✓ parking_location_clean.parquet
  Pad    : /Users/emilevandevoorde/Documents/mechelen_parking/data_intermediate/parking_location_clean.parquet
  Shape  : (11, 6)
  Kolommen: ['parking_id', 'longitude', 'latitude', 'total_capacity', 'parking_location_category', 'first_stable_month']

✓ MAD_shortterm.parquet
  Pad      : /Users/emilevandevoorde/Documents/mechelen_parking/data_processed/MAD_shortterm.parquet
  Shape    : (284524, 52)
  Grootte  : 4.5 MB
  Flags    : low_data_coverage, system_blackout, partial_year

✓ MAD_longterm.parquet
  Pad      : /Users/emilevandevoorde/Documents/mechelen_parking/data_processed/MAD_longterm.parquet
  Shape    : (46643, 51)
  Grootte  : 0.8 MB


In [27]:
print("=== FINALE INTEGRITEITCHECK ===\n")

checks = {
    "MAD_shortterm": (MAD_st, 284524),
    "MAD_longterm" : (MAD_lt,  46643),
}

for name, (df, expected_rows) in checks.items():
    print(f"--- {name} ---")

    # 1. Rijen intact
    ok1 = len(df) == expected_rows
    print(f"  Rijen correct ({expected_rows:,})          : {'✓' if ok1 else f'⚠️  {len(df):,}'}")

    # 2. Primaire sleutel uniek
    n_dupes = df.duplicated(subset=["parking_id", "rounded_hour"]).sum()
    print(f"  PK uniek                           : {'✓' if n_dupes == 0 else f'⚠️  {n_dupes} dupl.'}")

    # 3. Kernkolommen NaN-vrij
    core = ["parking_id", "rounded_hour", "occupancy_rate",
            "parking_location_category", "calendar_day_class"]
    core_nan = {c: int(df[c].isna().sum()) for c in core if c in df.columns}
    # calendar_day_class mag 5 NaN hebben (2018-12-31)
    core_ok  = all(v == 0 for k, v in core_nan.items()
                   if k != "calendar_day_class") and core_nan.get("calendar_day_class", 0) <= 5
    print(f"  Kernkolommen NaN-vrij              : {'✓' if core_ok else f'⚠️  {core_nan}'}")

    # 4. Flags aanwezig
    flags_ok = all(f in df.columns for f in
                   ["low_data_coverage","system_blackout","partial_year"])
    print(f"  Flags aanwezig                     : {'✓' if flags_ok else '⚠️  ontbrekend'}")

    # 5. Locatiecategorie volledig
    n_nan_cat = df["parking_location_category"].isna().sum()
    print(f"  parking_location_category NaN-vrij : {'✓' if n_nan_cat == 0 else f'⚠️  {n_nan_cat}'}")
    print()


=== FINALE INTEGRITEITCHECK ===

--- MAD_shortterm ---
  Rijen correct (284,524)          : ✓
  PK uniek                           : ✓
  Kernkolommen NaN-vrij              : ✓
  Flags aanwezig                     : ✓
  parking_location_category NaN-vrij : ✓

--- MAD_longterm ---
  Rijen correct (46,643)          : ✓
  PK uniek                           : ✓
  Kernkolommen NaN-vrij              : ✓
  Flags aanwezig                     : ✓
  parking_location_category NaN-vrij : ✓



## Samenvatting

### Outputbestanden

| Bestand | Locatie | Shape | Inhoud |
|---|---|---|---|
| `MAD_shortterm.parquet` | `data_processed/` | 284.524 × 54 | Alle shortterm-observaties met joins en flags |
| `MAD_longterm.parquet` | `data_processed/` | 46.643 × 53 | Alle longterm-observaties met joins en flags |
| `parking_location_clean.parquet` | `data_intermediate/` | 11 × 7 | Locatietabel met onboarding-info |

### Gedocumenteerde databeperkingen (voor methodologieparagraaf)

| Beperking | Omvang | Flag | Aanpak in modellering |
|---|---|---|---|
| Systeem-uitval jun 2021–jul 2022 | MNAR; sparse obs. geflagd | `system_blackout` | Uitsluiten uit training |
| Lage jaarlijkse dekking 2018, 2021, 2022 | 0–31% dekking | `low_data_coverage` | Uitsluiten uit training |
| Lopend jaar 2026 | 8.8% (t/m feb) | `partial_year` | Post-hoc validatie |
| Gefaseerde onboarding | 4 parkings stabiel pas vanaf 2024 | `first_stable_month` (ref.) | Sensitiviteitsanalyse per cohort |
| P Zandpoortvest absent | 622 pl. = 44% vestencap. | — | Beperking in thesis vermelden |

### Trainingsset definitie (aanbeveling)

| Subset | Filter | Rijen | Doel |
|---|---|---|---|
| Training | `year ∈ {2020,2023,2024}` + alle flags = 0 | ~130.000 | Modeltraining |
| Validatie | `year = 2024`, rolling window | — | Hyperparameter-tuning |
| **Holdout** | **`year = 2025`** | **87.600** | **Finale evaluatie** |

2025 als holdout is methodologisch sterk: 100% dekking, alle 10 parkings,
temporeel strikt gescheiden van training — wat temporele lekkage uitsluit
(Roberts et al., 2017; Bergmeir & Benítez, 2012).

### Volgende stap
➡️ `05_eda_temporal.ipynb` — Temporele patronenanalyse: uurlijkse,
dagelijkse en seizoenspatronen per parking en per categorie, als empirische
basis voor feature-ontwerp in notebook 08.

---
*Referenties:*
- Bergmeir, C. & Benítez, J.M. (2012). On the use of cross-validation for time series predictor evaluation.
- Little, R.J.A. & Rubin, D.B. (2002). Statistical Analysis with Missing Data (2nd ed.).
- Montero-Manso, P. & Hyndman, R.J. (2021). Principles and algorithms for forecasting groups of time series.
- Roberts, D.R. et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure.
- Saunders, J.A. et al. (2008). Missing data: Principles and practical guidelines.
- van Buuren, S. (2018). Flexible Imputation of Missing Data (2nd ed.).


In [28]:
# ════════════════════════════════════════════════════════════════
#  ALLERLAATSTE FULL CHECKUP — leest van disk, niet uit memory
# ════════════════════════════════════════════════════════════════

MAD_st_disk = pd.read_parquet(DATA_PROC / "MAD_shortterm.parquet")
MAD_lt_disk = pd.read_parquet(DATA_PROC / "MAD_longterm.parquet")
loc_disk    = pd.read_parquet(DATA_INT  / "parking_location_clean.parquet")

PASS, FAIL = "  ✓", "  ⚠️ "

def chk(label, condition, detail=""):
    status = PASS if condition else FAIL
    print(f"{status} {label}" + (f"  →  {detail}" if detail else ""))

print("=" * 62)
print("  CHECKUP A — Schijf vs. memory")
print("=" * 62)
chk("MAD_st: shape identiek aan memory",
    MAD_st_disk.shape == MAD_st.shape,
    f"disk={MAD_st_disk.shape}  mem={MAD_st.shape}")
chk("MAD_lt: shape identiek aan memory",
    MAD_lt_disk.shape == MAD_lt.shape,
    f"disk={MAD_lt_disk.shape}  mem={MAD_lt.shape}")
chk("MAD_st: kolommen identiek",
    list(MAD_st_disk.columns) == list(MAD_st.columns))
chk("loc: first_stable_month aanwezig",
    "first_stable_month" in loc_disk.columns)

print("\n" + "=" * 62)
print("  CHECKUP B — Interne consistentie rounded_hour ↔ date_only")
print("=" * 62)
# date_only moet gelijk zijn aan de datum van rounded_hour
# (geen off-by-one door DST of midnight edge)
derived_date = MAD_st_disk["rounded_hour"].dt.normalize()
stored_date  = pd.to_datetime(MAD_st_disk["date_only"]).dt.normalize()
n_mismatch   = (derived_date != stored_date).sum()
chk("rounded_hour.date == date_only (alle rijen)",
    n_mismatch == 0, f"{n_mismatch} mismatches")

if n_mismatch > 0:
    print("    Eerste mismatches:")
    mask = derived_date != stored_date
    print(MAD_st_disk[mask][["rounded_hour","date_only"]].head(5).to_string())

print("\n" + "=" * 62)
print("  CHECKUP C — Calendar join: spot-check bekende feestdagen")
print("=" * 62)
# Bekende Belgische feestdagen in de dataset
spot_holidays = {
    "2023-07-21": ("national_holiday", "national_holiday_name"),  # Nationale feestdag
    "2023-11-01": ("national_holiday", "national_holiday_name"),  # Allerheiligen
    "2023-12-25": ("national_holiday", "national_holiday_name"),  # Kerstmis
    "2023-05-01": ("national_holiday", "national_holiday_name"),  # Dag vd Arbeid
}
for date_str, (expected_class, name_col) in spot_holidays.items():
    ts   = pd.Timestamp(date_str)
    rows = MAD_st_disk[MAD_st_disk["date_only"] == ts]
    if len(rows) == 0:
        print(f"  ?  {date_str}  geen rijen in MAD (parking niet actief die dag)")
        continue
    actual_class = rows["calendar_day_class"].iloc[0]
    actual_name  = rows[name_col].iloc[0]
    ok = actual_class == expected_class
    chk(f"{date_str} = {expected_class}", ok,
        f"gevonden='{actual_class}'  naam='{actual_name}'")

# Spot-check schoolvakantie
vac_rows = MAD_st_disk[MAD_st_disk["date_only"] == pd.Timestamp("2023-07-03")]
if len(vac_rows) > 0:
    vac_class = vac_rows["calendar_day_class"].iloc[0]
    chk("2023-07-03 = school_vacation", vac_class == "school_vacation",
        f"gevonden='{vac_class}'")

print("\n" + "=" * 62)
print("  CHECKUP D — Fysische grenzen occupancy")
print("=" * 62)
for df, name in [(MAD_st_disk, "shortterm"), (MAD_lt_disk, "longterm")]:
    occ_min = df["occupancy_rate"].min()
    occ_max = df["occupancy_rate"].max()
    n_neg   = (df["occupancy_rate"] < 0).sum()
    n_over1 = (df["occupancy_rate"] > 1).sum()
    chk(f"{name}: occupancy_rate ≥ 0",  n_neg   == 0, f"min={occ_min:.4f}")
    chk(f"{name}: occupancy_rate ≤ 1",  n_over1 == 0, f"max={occ_max:.4f}  ({n_over1} rijen > 1)")
    n_spaces_neg = (df["number_of_occupied_spaces"] < 0).sum()
    chk(f"{name}: occupied_spaces ≥ 0", n_spaces_neg == 0,
        f"{n_spaces_neg} negatief")

print("\n" + "=" * 62)
print("  CHECKUP E — Flag-waarden: alleen 0 en 1")
print("=" * 62)
for flag in ["low_data_coverage", "system_blackout", "partial_year"]:
    vals = set(MAD_st_disk[flag].unique())
    chk(f"shortterm.{flag} ∈ {{0,1}}", vals <= {0, 1}, f"gevonden: {vals}")
for flag in ["low_data_coverage", "system_blackout", "partial_year"]:
    vals = set(MAD_lt_disk[flag].unique())
    chk(f"longterm.{flag}  ∈ {{0,1}}", vals <= {0, 1}, f"gevonden: {vals}")

print("\n" + "=" * 62)
print("  CHECKUP F — Blackout-periode correct afgebakend")
print("=" * 62)
# Mei 2021 moet NIET geflagd zijn, juni 2021 WEL
may21  = MAD_st_disk[MAD_st_disk["rounded_hour"] == pd.Timestamp("2021-05-31 23:00")]
jun21  = MAD_st_disk[MAD_st_disk["rounded_hour"] == pd.Timestamp("2021-06-01 00:00")]
jun22  = MAD_st_disk[MAD_st_disk["rounded_hour"] == pd.Timestamp("2022-06-30 23:00")]
jul22  = MAD_st_disk[MAD_st_disk["rounded_hour"] == pd.Timestamp("2022-07-01 00:00")]

for label, subset, expected in [
    ("2021-05-31 23:00 → blackout=0", may21, 0),
    ("2021-06-01 00:00 → blackout=1", jun21, 1),
    ("2022-06-30 23:00 → blackout=1", jun22, 1),
    ("2022-07-01 00:00 → blackout=0", jul22, 0),
]:
    if len(subset) == 0:
        print(f"  ?  {label}  (geen rijen op dit tijdstip — normaal)")
        continue
    actual = subset["system_blackout"].iloc[0]
    chk(label, actual == expected, f"gevonden={actual}")

print("\n" + "=" * 62)
print("  CHECKUP G — Overlap shortterm ∩ longterm (2024)")
print("=" * 62)
# Longterm loopt enkel 2024; check of dezelfde parkings in beide zitten
st_2024_pids = set(MAD_st_disk[MAD_st_disk["year"]==2024]["parking_id"].unique())
lt_2024_pids = set(MAD_lt_disk["parking_id"].unique())
overlap_pids = st_2024_pids & lt_2024_pids
print(f"  Shortterm 2024 parkings : {sorted(st_2024_pids)}")
print(f"  Longterm  parkings      : {sorted(lt_2024_pids)}")
print(f"  Overlap (beide datasets): {sorted(overlap_pids)}")
print(f"  → {len(overlap_pids)} parkings aanwezig in BEIDE datasets in 2024")
print(f"    (bij modellering: kies expliciet shortterm OF longterm per parking)")

print("\n" + "=" * 62)
print("  CHECKUP H — Trainingsset definitie (finale telling)")
print("=" * 62)
exclude = (
    (MAD_st_disk["low_data_coverage"] == 1) |
    (MAD_st_disk["system_blackout"]   == 1) |
    (MAD_st_disk["partial_year"]      == 1) |
    (MAD_st_disk["year"]              <  2020)
)
train    = ~exclude & (MAD_st_disk["year"] != 2025)
holdout  = MAD_st_disk["year"] == 2025

print(f"  Totaal MAD shortterm : {len(MAD_st_disk):>7,}")
print(f"  Uitgesloten          : {exclude.sum():>7,}  ({exclude.mean()*100:.1f}%)")
print(f"  Trainingsset         : {train.sum():>7,}  ({train.mean()*100:.1f}%)")
print(f"  Holdout 2025         : {holdout.sum():>7,}  ({holdout.mean()*100:.1f}%)")
print(f"  Trainings-jaren      : {sorted(MAD_st_disk[train]['year'].unique())}")
chk("Train + holdout + excl = totaal",
    train.sum() + holdout.sum() + exclude.sum() == len(MAD_st_disk))

print("\n" + "=" * 62)
print("  EINDOORDEEL")
print("=" * 62)


  CHECKUP A — Schijf vs. memory
  ✓ MAD_st: shape identiek aan memory  →  disk=(284524, 52)  mem=(284524, 52)
  ✓ MAD_lt: shape identiek aan memory  →  disk=(46643, 51)  mem=(46643, 51)
  ✓ MAD_st: kolommen identiek
  ✓ loc: first_stable_month aanwezig

  CHECKUP B — Interne consistentie rounded_hour ↔ date_only
  ⚠️  rounded_hour.date == date_only (alle rijen)  →  10 mismatches
    Eerste mismatches:
       rounded_hour  date_only
40195    2026-02-02 2026-02-01
76844    2026-02-02 2026-02-01
99327    2026-02-02 2026-02-01
121506   2026-02-02 2026-02-01
147845   2026-02-02 2026-02-01

  CHECKUP C — Calendar join: spot-check bekende feestdagen
  ✓ 2023-07-21 = national_holiday  →  gevonden='national_holiday'  naam='Nationale feestdag 2023'
  ✓ 2023-11-01 = national_holiday  →  gevonden='national_holiday'  naam='Allerheiligen 2023'
  ✓ 2023-12-25 = national_holiday  →  gevonden='national_holiday'  naam='1e Kerstdag 2023'
  ✓ 2023-05-01 = national_holiday  →  gevonden='national_holiday'  